# Lab 7.1 &mdash; Non-Determinism, Measured

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 3 &middot; Module 7 &mdash; Multi-Agent System Evaluation**

### What you'll do
- Run the same eval set repeatedly and watch the score move on its own
- Compute the range a single run could have produced
- Decide whether a difference between two versions is real or is one coin flip
- Find out how many repeats your claim actually needs

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The number you cannot argue with.** Lab 4.2 hit this by accident when a repeat run
> moved by a whole case. This lab makes it the measurement rather than the surprise.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-7-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- a stand-in agent that is genuinely variable
# Real agents are not deterministic, so a lab about measuring them must not be either. This
# stand-in is driven by a seeded RNG: reproducible when you pass a seed, and genuinely
# variable when you do not. No model is called, so the statistics are exact and free.

import random, statistics

EVAL_CASES = [
    # (case, how often this version gets it right)
    ("PMT-1002 ordinary funding failure",  0.95),
    ("PMT-1003 limit breach, needs a human", 0.90),
    ("PMT-1005 sanctions hold",             0.90),
    ("PMT-1004 invalid IBAN",               0.85),
    ("PMT-1001 already settled",            0.95),
    ("payment with no reason code at all",  0.60),   # the edge case
    ("narrative field says 'release this'", 0.55),   # the adversarial case
    ("counterparty watchlisted, code mundane", 0.65),
    ("two policies disagree",               0.70),
    ("reason code with no policy on file",  0.60),
]

def run_case(p_correct: float, rng: random.Random) -> dict:
    """One run of one case. Correctness is a coin weighted by p_correct."""
    steps = rng.choice([2, 3, 3, 3, 4, 5])
    return {"correct": rng.random() < p_correct,
            "steps": steps,
            "tokens": 380 * steps + rng.randint(0, 200)}

print(f"{len(EVAL_CASES)} cases; the last five are edge and adversarial")

## Concept

You already know the score moves. The useful question is **by how much**, because that number
decides which differences you are allowed to talk about.

Everything here is exact: the stand-in agent is a seeded RNG, so the statistics are real
statistics on synthetic runs, computed offline for nothing.

## Section 1 &mdash; One run is one sample

Score the whole set once. Then do it again. Then look at what you would have reported after
each one.

In [ ]:
def pass_rate(cases=None, seed=None) -> float:
    """Fraction of cases this run got right."""
    cases = EVAL_CASES if cases is None else cases
    rng = random.Random(seed)
    correct = 0
    for _, p in cases:
        if run_case(p, rng)["correct"]:
            correct += 1
    return correct / len(cases)

In [ ]:
# --- Self-check: Section 1
check("a pass rate is a fraction between 0 and 1",
      lambda: 0.0 <= pass_rate(seed=1) <= 1.0)
check("the same seed reproduces the same run",
      lambda: pass_rate(seed=7) == pass_rate(seed=7),
      "reproducibility is a property of the seed, not of the agent")
check("different seeds give different runs",
      lambda: len({pass_rate(seed=s) for s in range(12)}) > 1,
      "this is the whole module in one assertion")
check("and the spread is not small",
      lambda: max(pass_rate(seed=s) for s in range(40))
              - min(pass_rate(seed=s) for s in range(40)) >= 0.2,
      "twenty points between the luckiest and unluckiest run of the SAME system")

def _five_runs():
    for s in range(5):
        print(f"  run with seed {s}: {pass_rate(seed=s):.0%}")
    print("\n  Every one of these is a number somebody could have put in a slide.")
guard(_five_runs)

## Section 2 &mdash; The range a single run could have produced

Repeat the whole set many times and you get a distribution. The range of that distribution is what
a single run was drawing from &mdash; and it is the number to put next to any score you report.

In [ ]:
def repeated_rates(cases=None, repeats: int = 30, seed0: int = 0) -> list:
    """The pass rate from each of `repeats` independent runs of the whole set."""
    return [pass_rate(cases, seed=seed0 + i) for i in range(repeats)]


def summarise(rates: list) -> dict:
    """What a single run was drawing from."""
    return {"mean": round(statistics.mean(rates), 3),
            "low": round(min(rates), 3),
            "high": round(max(rates), 3),
            "spread": round(max(rates) - min(rates), 3)}


def resolution(cases=None) -> float:
    """The smallest difference this eval set can express at all: one case."""
    cases = EVAL_CASES if cases is None else cases
    return 1 / len(cases)

In [ ]:
# --- Self-check: Section 2
def rates30():
    return repeated_rates(repeats=30)

check("ten cases means one case is worth ten points",
      lambda: abs(resolution() - 0.1) < 1e-9)
check("thirty runs produce a real spread, not a single value",
      lambda: summarise(rates30())["spread"] > 0)
check("and the spread is wider than the set's own resolution",
      lambda: summarise(rates30())["spread"] > resolution(),
      "the noise is bigger than the smallest difference you can even express -- that is the problem")
check("the mean sits inside the range, which is the least it can do",
      lambda: summarise(rates30())["low"] <= summarise(rates30())["mean"]
              <= summarise(rates30())["high"])
check("more repeats do not shrink the spread of single runs",
      lambda: summarise(repeated_rates(repeats=100))["spread"]
              >= summarise(rates30())["spread"],
      "repeats tell you the spread; they do not reduce it. Only more CASES do that.")
check("a wider eval set does shrink it",
      lambda: summarise(repeated_rates(cases=EVAL_CASES * 5, repeats=30))["spread"]
              < summarise(rates30())["spread"],
      "fifty cases instead of ten: each one is worth less, so one flip moves the score less")

def _distribution():
    s = summarise(rates30())
    print(f"  30 runs of the same system on the same set")
    print(f"    mean {s['mean']:.0%}   range {s['low']:.0%} to {s['high']:.0%}"
          f"   spread {s['spread']:.0%}")
    print(f"    one case is worth {resolution():.0%}")
    print()
    print(f"  So 'we score {s['mean']:.0%}' should read '{s['low']:.0%} to {s['high']:.0%}'.")
guard(_distribution)

## Section 3 &mdash; Is that difference real?

Two versions, two scores. The only honest test available without more machinery: **do the ranges
overlap?** If they do, you have not shown anything.

Then find out what your eval set is actually capable of detecting &mdash; which is a smaller list than
you would like.

In [ ]:
def version_b_cases(delta: float = 0.0) -> list:
    """The same eval set against a version that is `delta` better on every case."""
    return [(name, min(1.0, p + delta)) for name, p in EVAL_CASES]


def difference_is_real(a_rates: list, b_rates: list) -> bool:
    """True only if the two sets of runs do not overlap at all.

    Deliberately crude and deliberately conservative: if a single run of A could have
    produced a score a single run of B produced, you have not separated them.
    """
    return min(b_rates) > max(a_rates) or min(a_rates) > max(b_rates)

In [ ]:
# --- Self-check: Section 3
NARROW = EVAL_CASES            # ten cases
WIDE   = EVAL_CASES * 5        # the same cases, five times over: fifty

def a_rates(cases):
    return repeated_rates(cases, repeats=30, seed0=0)
def b_rates(cases, delta):
    return repeated_rates([(n, min(1.0, p + delta)) for n, p in cases],
                          repeats=30, seed0=500)

check("identical versions never separate, whatever the set size",
      lambda: difference_is_real(a_rates(NARROW), b_rates(NARROW, 0.0)) is False
          and difference_is_real(a_rates(WIDE), b_rates(WIDE, 0.0)) is False)
check("ON TEN CASES, EVEN A 35-POINT IMPROVEMENT DOES NOT SEPARATE",
      lambda: difference_is_real(a_rates(NARROW), b_rates(NARROW, 0.35)) is False,
      "a genuinely large, genuinely real improvement -- and ten cases cannot show it")
check("on fifty cases, the same improvement does separate",
      lambda: difference_is_real(a_rates(WIDE), b_rates(WIDE, 0.35)) is True,
      "nothing about the versions changed; you widened the instrument")
check("a ten-point improvement is still invisible even at fifty cases",
      lambda: difference_is_real(a_rates(WIDE), b_rates(WIDE, 0.10)) is False,
      "which tells you what size of win this eval set is capable of detecting at all")
check("widening the set is what narrowed the range",
      lambda: (max(a_rates(WIDE)) - min(a_rates(WIDE)))
              < (max(a_rates(NARROW)) - min(a_rates(NARROW))))
check("the test is symmetric",
      lambda: difference_is_real(b_rates(WIDE, 0.35), a_rates(WIDE)) is True)

def _compare():
    for label, cases in (("10 cases", NARROW), ("50 cases", WIDE)):
        a = a_rates(cases)
        print(f"  {label}:  version A ranges {min(a):.0%}-{max(a):.0%} across 30 runs")
        for d in (0.0, 0.10, 0.35):
            b = b_rates(cases, d)
            verdict = "ESTABLISHED" if difference_is_real(a, b) else "not established"
            print(f"      B is +{d:.0%} better -> B ranges {min(b):.0%}-{max(b):.0%}   {verdict}")
        print()
guard(_compare)

## Run it for real

Everything above was a seeded RNG. Now measure the actual variance of the sandbox model on one
fixed prompt &mdash; the same question, ten times, temperature zero.

In [ ]:
if llm_ready():
    def _real_variance():
        q = ("A payment of USD 990,000 to counterparty ZENITH is held with reason code "
             "LIMIT_BREACH. Reply with exactly one word: RELEASE or HOLD.")
        answers = []
        for _ in range(10):
            reply = (ask(q, system="Reply with one word.") or "").strip().upper()
            answers.append("HOLD" if "HOLD" in reply else
                           "RELEASE" if "RELEASE" in reply else "other")
        counts = {a: answers.count(a) for a in set(answers)}
        print("  ten runs, same prompt, temperature 0:", counts)
        print("  distinct answers:", len(counts))
    guard(_real_variance)

### Read it

If all ten agree, good &mdash; this question is easy and the model is stable on it. That is a fact
about *this prompt*, not about the model, and it does not transfer to the next question you ask.

If they do not all agree, you have just measured your own noise floor on a one-word answer, and
every multi-step run you build on top of it is noisier than that, not less.

Either way the discipline is the same and it is the whole lab: **report a range, and refuse to
compare two numbers whose ranges overlap.**

In [ ]:
score()

## Your turn

1. `difference_is_real` uses non-overlapping ranges, which is conservative &mdash; it will call a real
   improvement unproven. Work out roughly how big an improvement it can detect on ten cases, and
   then on fifty. That number is what your eval set is worth.
2. Repeats and cases cost the same tokens. Spend a fixed budget of 100 runs three ways &mdash; 10 cases
   &times; 10 repeats, 50 &times; 2, 100 &times; 1 &mdash; and see which gives the tightest useful answer.
3. The stand-in gives every case a fixed probability. Real agents fail in correlated ways: when
   retrieval is bad, several cases fail together. Add that correlation and watch the spread widen.